# Kaggle HGT Training From Existing Three-Tier Graph

Use this notebook when the three-tier graph artifact already exists and has been uploaded to Kaggle.

Flow:

```text
GitHub repo clone
-> find graph_artifact_3tier_t082_k5.npz + .meta.json in /kaggle/input
-> convert NPZ to graph_store_mmap_csr_v1, layout=numpy_memmap_csr
-> patch HGT config to source=graph_store and batch_mode=neighbor_sampling
-> train HGT
```

This skips teacher/student/1D-CNN and graph rebuild. Graph growth control at this stage is the mmap/CSR store plus neighbor sampling. Threshold/top-k graph pruning is already baked into the uploaded NPZ.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
from pathlib import Path
import queue
import shutil
import subprocess
import sys
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any

import torch

# Code is expected to be current on GitHub.
GITHUB_REPO_URL = "https://github.com/LeThanhPhat-ATTT2023/Do-an-chuyen-nganh_NT114.git"
GITHUB_BRANCH = ""
FORCE_RECLONE = False
GITHUB_PULL_IF_CLONED = True    # pull latest code when WORK_DIR is a git clone

WORK_DIR = Path("/kaggle/working/nt114_hgt_existing_graph")
RESULT_ZIP = Path("/kaggle/working/hgt_existing_graph_results.zip")

GRAPH_NPZ_NAME = "graph_artifact_3tier.npz"
GRAPH_META_NAME = "graph_artifact_3tier.meta.json"
TRAIN_GRAPH_STORE_DIR_NAME = "graph_store_mmap_csr_v1"

# deployment = one production HGT config. paper_variants = baseline + variants in parallel.
HGT_RUN_MODE = "deployment"

# Auto-detect all available GPUs at startup; works on any machine (1 GPU, 2 GPU, 4 GPU, ...)
GPU_IDS: list[int] = list(range(torch.cuda.device_count())) or [0]
MAX_PARALLEL_HGT_RUNS: int = len(GPU_IDS)   # will be capped to len(RUNS) later
SAFE_15GB_PROFILE = True
MAX_BATCH_SEED_FLOWS = 128
MIN_BATCH_SEED_FLOWS = 16
OOM_RETRIES = 3
ALLOW_CPU_FALLBACK = False

INSTALL_WITH_DEPS = False
INSTALL_MISSING_DEPS = True
GRAPH_PKT_SEM_NPY_NAME = "graph_artifact_3tier_packet_semantic_x.npy"
GRAPH_PKT_SEM_NPY_ZST_NAME = "graph_artifact_3tier_packet_semantic_x.npy.zst"
# Decompressed sidecar goes to /tmp (not counted against the 19 GB /kaggle/working quota).
GRAPH_PKT_SEM_DECOMPRESS_DIR = Path("/tmp")
COPY_GRAPH_TO_WORK_DIR = False
USE_INPUT_TRAIN_GRAPH_STORE_IF_PRESENT = True
RESET_OUTPUTS = False

DEPLOYMENT_RUNS = [
    {"name": "deployment_t082_k5_l3_d01", "config": "configs/hgt_t082_k5_l3_d01.yaml"},
]

PAPER_VARIANT_RUNS = [
    {"name": "baseline_t082_k5_l3_d01", "config": "configs/hgt_t082_k5_l3_d01.yaml"},
    {"name": "xgnid_dual_modal_l1_h32_h4", "config": "configs/hgt_paper_variants/hgt_t082_k5_xgnid_dual_modal_l1_h32_h4.yaml"},
    {"name": "one2_iov_l1_h64_h2", "config": "configs/hgt_paper_variants/hgt_t082_k5_one2_iov_l1_h64_h2.yaml"},
    {"name": "relgt_multi_token_l3_h128_h8", "config": "configs/hgt_paper_variants/hgt_t082_k5_relgt_multi_token_l3_h128_h8.yaml"},
    {"name": "gatransformer_deep_l6_h256_h8", "config": "configs/hgt_paper_variants/hgt_t082_k5_gatransformer_deep_l6_h256_h8.yaml"},
    {"name": "ahgt_dfd_funnel_l3_h128_h4", "config": "configs/hgt_paper_variants/hgt_t082_k5_ahgt_dfd_funnel_l3_h128_h4.yaml"},
    {"name": "dlg_ids_sparse_l2_h128_h4", "config": "configs/hgt_paper_variants/hgt_t082_k5_dlg_ids_sparse_l2_h128_h4.yaml"},
]

RUNS = DEPLOYMENT_RUNS if HGT_RUN_MODE == "deployment" else PAPER_VARIANT_RUNS


In [ ]:
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Kaggle: Settings -> Accelerator -> GPU T4 x2.")

for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f"GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.1f} GB")

# GPU_IDS is already auto-detected from torch.cuda.device_count() in the config cell.
# Cap MAX_PARALLEL_HGT_RUNS to the number of available runs to avoid idle workers.
MAX_PARALLEL_HGT_RUNS = min(MAX_PARALLEL_HGT_RUNS, len(GPU_IDS), len(RUNS))
print("HGT_RUN_MODE:", HGT_RUN_MODE)
print("Active GPU_IDS:", GPU_IDS)
print("Max parallel HGT runs:", MAX_PARALLEL_HGT_RUNS)


In [ ]:
def run_cmd(cmd: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print("\n$", " ".join(str(x) for x in cmd))
    subprocess.check_call([str(x) for x in cmd], cwd=cwd or WORK_DIR, env=env)


def is_repo_root(path: Path) -> bool:
    return (
        (path / "src" / "graphslm_ids").exists()
        and (path / "configs" / "hgt_t082_k5_l3_d01.yaml").exists()
        and (path / "pyproject.toml").exists()
    )


def find_repo_in_kaggle_input() -> Path | None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for root in sorted(input_root.glob("*")):
        if is_repo_root(root):
            return root
        for config_path in root.rglob("configs/hgt_t082_k5_l3_d01.yaml"):
            candidate = config_path.parents[1]
            if is_repo_root(candidate):
                return candidate
    return None


def prepare_repo() -> None:
    if FORCE_RECLONE and WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    if is_repo_root(WORK_DIR):
        if GITHUB_PULL_IF_CLONED and (WORK_DIR / ".git").exists():
            print("Git pull latest code:", WORK_DIR)
            try:
                subprocess.check_call(["git", "pull", "--ff-only"], cwd=WORK_DIR)
            except subprocess.CalledProcessError as exc:
                print(f"[warn] git pull failed ({exc}), continuing with existing code.")
        else:
            print("Repo already prepared:", WORK_DIR)
        return
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)

    source_repo = find_repo_in_kaggle_input()
    if source_repo is not None:
        print("Copy repo:", source_repo, "->", WORK_DIR)
        shutil.copytree(source_repo, WORK_DIR, ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", ".pytest_cache"))
        return

    cmd = ["git", "clone"]
    if GITHUB_BRANCH.strip():
        cmd += ["--branch", GITHUB_BRANCH]
    cmd += [GITHUB_REPO_URL, str(WORK_DIR)]
    print("Clone repo:", " ".join(cmd))
    subprocess.check_call(cmd)


def import_ok(module_name: str) -> bool:
    try:
        __import__(module_name)
        return True
    except Exception:
        return False


def install_repo() -> None:
    missing = [name for name in ["numpy", "pandas", "yaml", "torch", "tqdm"] if not import_ok(name)]
    if missing:
        print("Missing deps:", missing)
        if not INSTALL_MISSING_DEPS:
            raise RuntimeError("Missing dependencies and INSTALL_MISSING_DEPS=False")
        run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements-ml.txt"])

    try:
        import zstandard  # noqa: F401
    except ImportError:
        run_cmd([sys.executable, "-m", "pip", "install", "--quiet", "zstandard"])

    cmd = [sys.executable, "-m", "pip", "install", "-e", "."]
    if not INSTALL_WITH_DEPS:
        cmd.append("--no-deps")
    run_cmd(cmd)


prepare_repo()
os.chdir(WORK_DIR)
if RESET_OUTPUTS and (WORK_DIR / "outputs").exists():
    shutil.rmtree(WORK_DIR / "outputs")
install_repo()
print("CWD:", Path.cwd())


In [ ]:
PROCESSED_DIR = WORK_DIR / "data" / "processed"
GRAPH_NPZ_LOCAL = PROCESSED_DIR / GRAPH_NPZ_NAME
GRAPH_META_LOCAL = PROCESSED_DIR / GRAPH_META_NAME
# Graph store goes to /tmp — not counted against the 19.5 GB /kaggle/working quota.
# Packet features.f32 alone can be 50-60 GB, far exceeding the working dir limit.
TRAIN_GRAPH_STORE_ROOT = Path("/tmp") / TRAIN_GRAPH_STORE_DIR_NAME
RUNTIME_CONFIG_DIR = WORK_DIR / "outputs" / "kaggle_existing_graph_configs"
LOG_DIR = WORK_DIR / "outputs" / "hgt_existing_graph_logs"
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


def find_input_file(name: str) -> Path:
    candidates = [WORK_DIR / "data" / "processed" / name]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates += sorted(input_root.rglob(name))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing {name}. Upload it as a Kaggle Dataset.")


def prepare_graph_artifact() -> tuple[Path, Path]:
    graph_npz = find_input_file(GRAPH_NPZ_NAME)
    graph_meta = find_input_file(GRAPH_META_NAME)
    if not COPY_GRAPH_TO_WORK_DIR:
        print("Use graph NPZ in place:", graph_npz)
        print("Use graph meta in place:", graph_meta)
        return graph_npz, graph_meta
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    if graph_npz.resolve() != GRAPH_NPZ_LOCAL.resolve():
        shutil.copy2(graph_npz, GRAPH_NPZ_LOCAL)
    if graph_meta.resolve() != GRAPH_META_LOCAL.resolve():
        shutil.copy2(graph_meta, GRAPH_META_LOCAL)
    return GRAPH_NPZ_LOCAL, GRAPH_META_LOCAL


GRAPH_NPZ, GRAPH_META_JSON = prepare_graph_artifact()
print("GRAPH_NPZ:", GRAPH_NPZ)
print("GRAPH_META_JSON:", GRAPH_META_JSON)


def find_packet_semantic_npy(graph_npz_path: Path, graph_meta_json_path: Path) -> Path | None:
    """Locate the packet_semantic_x.npy (or .npy.zst / .npy.gz / .npy.zip) sidecar alongside the graph NPZ."""
    try:
        meta = json.loads(graph_meta_json_path.read_text(encoding="utf-8"))
        meta_path = meta.get("packet_semantic_x_npy")
        if meta_path and Path(meta_path).exists():
            return Path(meta_path)
    except Exception:
        pass
    stem = graph_npz_path.stem
    for suffix in (
        "_packet_semantic_x.npy",
        "_packet_semantic_x.npy.zst",
        "_packet_semantic_x.npy.gz",
        "_packet_semantic_x.npy.zip",
    ):
        sidecar = graph_npz_path.with_name(stem + suffix)
        if sidecar.exists():
            return sidecar
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for name in (GRAPH_PKT_SEM_NPY_ZST_NAME, GRAPH_PKT_SEM_NPY_NAME):
            for candidate in sorted(input_root.rglob(name)):
                return candidate
        for pattern in (
            "*_packet_semantic_x.npy.zst",
            "*_packet_semantic_x.npy.gz",
            "*_packet_semantic_x.npy.zip",
            "*_packet_semantic_x.npy",
        ):
            for candidate in sorted(input_root.rglob(pattern)):
                return candidate
    return None


def decompress_packet_semantic_npy(found_path: Path) -> Path:
    """Return a plain .npy path, decompressing .npy.zst / .npy.gz / .npy.zip if needed.

    The decompressed file is written to GRAPH_PKT_SEM_DECOMPRESS_DIR (/tmp by default)
    so it does not consume the 19 GB /kaggle/working quota.
    """
    name = found_path.name
    out_dir = GRAPH_PKT_SEM_DECOMPRESS_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    if name.endswith(".npy.zst"):
        import zstandard as zstd
        out_path = out_dir / name[:-4]                # strip .zst -> .npy
        if not out_path.exists():
            print(f"Decompressing (zstd) {found_path} -> {out_path}")
            dctx = zstd.ZstdDecompressor()
            with found_path.open("rb") as f_in, out_path.open("wb") as f_out:
                dctx.copy_stream(f_in, f_out)
        return out_path
    if name.endswith(".npy.gz"):
        import gzip
        out_path = out_dir / name[:-3]                # strip .gz -> .npy
        if not out_path.exists():
            print(f"Decompressing (gzip) {found_path} -> {out_path}")
            with gzip.open(found_path, "rb") as f_in, out_path.open("wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
        return out_path
    if name.endswith(".npy.zip"):
        import zipfile as _zf
        out_path = out_dir / name[:-4]                # strip .zip -> .npy
        if not out_path.exists():
            print(f"Extracting (zip) {found_path} -> {out_path}")
            with _zf.ZipFile(found_path, "r") as zf:
                members = [m for m in zf.namelist() if m.endswith(".npy")]
                if not members:
                    raise RuntimeError(f"No .npy entry found inside {found_path}")
                with zf.open(members[0]) as src, out_path.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
        return out_path
    return found_path


_raw_pkt_sem = find_packet_semantic_npy(GRAPH_NPZ, GRAPH_META_JSON)
if _raw_pkt_sem:
    GRAPH_PKT_SEM_NPY = decompress_packet_semantic_npy(_raw_pkt_sem)
    print("GRAPH_PKT_SEM_NPY:", GRAPH_PKT_SEM_NPY, "(source:", _raw_pkt_sem, ")")
else:
    GRAPH_PKT_SEM_NPY = None
    print("[warn] packet_semantic_x.npy not found; conversion will fail if NPZ lacks embedded packet_semantic_x.")


In [ ]:
def graph_store_layout(root: Path) -> str | None:
    manifest = root / "manifest.json"
    if not manifest.exists():
        return None
    return str(json.loads(manifest.read_text(encoding="utf-8")).get("layout", ""))


def is_training_graph_store(root: Path) -> bool:
    return graph_store_layout(root) == "numpy_memmap_csr"


def find_input_training_graph_store() -> Path | None:
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for manifest in sorted(input_root.rglob("manifest.json")):
        root = manifest.parent
        if is_training_graph_store(root):
            return root
    return None


def build_training_graph_store() -> Path:
    if is_training_graph_store(TRAIN_GRAPH_STORE_ROOT):
        print("Use local mmap/CSR graph store:", TRAIN_GRAPH_STORE_ROOT)
        return TRAIN_GRAPH_STORE_ROOT
    legacy_store = WORK_DIR / "data" / "graph_store_v1"
    legacy_layout = graph_store_layout(legacy_store)
    if legacy_layout and legacy_layout != "numpy_memmap_csr":
        print("Ignore runtime graph_store_v1 layout:", legacy_layout)
    input_store = find_input_training_graph_store() if USE_INPUT_TRAIN_GRAPH_STORE_IF_PRESENT else None
    if input_store is not None:
        print("Use input mmap/CSR graph store:", input_store)
        return input_store
    cmd = [
        sys.executable, "-u", "-m", "graphslm_ids.offline_path.training.on_disk_graph_store",
        "--graph-npz", str(GRAPH_NPZ),
        "--graph-meta-json", str(GRAPH_META_JSON),
        "--output-root", str(TRAIN_GRAPH_STORE_ROOT),
    ]
    if GRAPH_PKT_SEM_NPY is not None:
        cmd += ["--packet-semantic-npy", str(GRAPH_PKT_SEM_NPY)]
        # Symlink instead of copy to avoid writing ~55 GB to /tmp (Kaggle disk limit).
        cmd += ["--symlink-packet-features"]
    run_cmd(cmd)
    if not is_training_graph_store(TRAIN_GRAPH_STORE_ROOT):
        raise RuntimeError("Converted graph store must have layout=numpy_memmap_csr")
    return TRAIN_GRAPH_STORE_ROOT


GRAPH_STORE_ROOT = build_training_graph_store()
print("GRAPH_STORE_ROOT:", GRAPH_STORE_ROOT)
print("GRAPH_STORE_LAYOUT:", graph_store_layout(GRAPH_STORE_ROOT))
manifest = json.loads((GRAPH_STORE_ROOT / "manifest.json").read_text(encoding="utf-8"))
print("NODE_COUNTS:", manifest.get("node_counts"))
print("EDGE_COUNTS:", manifest.get("edge_counts"))


In [ ]:
import yaml


def load_yaml(path: Path) -> dict[str, Any]:
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}


def dump_yaml(path: Path, data: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=False), encoding="utf-8")


def patch_hgt_config(run_cfg: dict[str, str], retry: int = 0) -> tuple[Path, dict[str, Any]]:
    cfg = load_yaml(WORK_DIR / run_cfg["config"])
    data = cfg.setdefault("data", {})
    data.update(
        source="graph_store",
        graph_npz=str(GRAPH_NPZ),
        graph_meta_json=str(GRAPH_META_JSON),
        graph_store_root=str(GRAPH_STORE_ROOT),
        read_sealed_only=True,
        packet_feature=data.get("packet_feature", "semantic"),
        add_reverse_edges=True,
        standardize_flow_features=True,
        use_semantic_edge_weights=True,
    )
    train = cfg.setdefault("train", {})
    train.update(batch_mode="neighbor_sampling", device="cuda", amp=True, activation_checkpointing=True)
    train["monitor"] = train.get("monitor", "val_macro_f1")
    train["log_every"] = train.get("log_every", 1)
    base_batch = int(train.get("batch_seed_flows", 256))
    base_grad = int(train.get("grad_accum_steps", 1))
    cap = min(base_batch, MAX_BATCH_SEED_FLOWS) if SAFE_15GB_PROFILE else base_batch
    batch = max(MIN_BATCH_SEED_FLOWS, cap // (2 ** retry))
    train["batch_seed_flows"] = int(batch)
    train["grad_accum_steps"] = max(base_grad, math.ceil((base_batch * base_grad) / batch))

    sampler = cfg.setdefault("sampler", {})
    sampler.setdefault("hops", None)
    sampler.setdefault("fanouts", {
        "flow__contains__packet": 20,
        "packet__next_packet__packet": 4,
        "packet__matches_technique__technique": 5,
        "flow__matches_technique__technique": 5,
        "technique__belongs_to_tactic__tactic": 1,
    })
    sampler.setdefault("reverse_fanouts", {"rev_contains": 1, "rev_next_packet": 1, "rev_matches_technique": 0, "rev_belongs_to_tactic": 0})
    sampler.setdefault("always_include_all_tactics", True)
    sampler.setdefault("always_include_all_techniques", True)

    dataloader = cfg.setdefault("dataloader", {})
    n_cpu = max(1, (os.cpu_count() or 2) // max(1, len(GPU_IDS)))
    dataloader.update(
        num_workers=min(int(dataloader.get("num_workers", 4)), min(4, n_cpu)),
        prefetch_factor=int(dataloader.get("prefetch_factor", 2)),
        pin_memory=True,
        persistent_workers=True,
    )

    suffix = "" if retry == 0 else f"_retry{retry}"
    out_path = RUNTIME_CONFIG_DIR / f"{run_cfg['name']}{suffix}.yaml"
    dump_yaml(out_path, cfg)
    return out_path, cfg


def summary_path(config_path: Path) -> Path:
    cfg = load_yaml(config_path)
    return WORK_DIR / cfg["train"]["output_dir"] / "training_summary.json"


for run_cfg in RUNS:
    cfg_path, cfg = patch_hgt_config(run_cfg, retry=0)
    t = cfg["train"]
    print(run_cfg["name"], cfg_path, "batch_mode=", t["batch_mode"], "batch_seed_flows=", t["batch_seed_flows"], "grad_accum_steps=", t["grad_accum_steps"])


In [ ]:
OOM_MARKERS = ("CUDA out of memory", "torch.OutOfMemoryError", "CUBLAS_STATUS_ALLOC_FAILED", "CUDA error: out of memory")


def log_has_oom(path: Path) -> bool:
    return path.exists() and any(marker in path.read_text(encoding="utf-8", errors="ignore") for marker in OOM_MARKERS)


def train_process(run_cfg: dict[str, str], config_path: Path, gpu_id: int, retry: int) -> tuple[int, Path]:
    log_path = LOG_DIR / f"{run_cfg['name']}_gpu{gpu_id}_retry{retry}.log"
    env = os.environ.copy()
    env.update(
        PYTHONUNBUFFERED="1",
        PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True,max_split_size_mb:128",
        CUDA_VISIBLE_DEVICES=str(gpu_id),
        OMP_NUM_THREADS="2",
        MKL_NUM_THREADS="2",
    )
    cmd = [sys.executable, "-u", "-m", "graphslm_ids.offline_path.training.train_hgt_flow_classifier", "--config", str(config_path), "--device", "cuda"]
    print(f"\n=== TRAIN {run_cfg['name']} gpu={gpu_id} retry={retry} ===")
    print("log:", log_path)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(cmd, cwd=WORK_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        assert process.stdout is not None
        for line in process.stdout:
            print(f"[gpu{gpu_id}:{run_cfg['name']}] {line}", end="", flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    return code, log_path


def train_one(run_cfg: dict[str, str], gpu_id: int) -> dict[str, Any]:
    first_config, _ = patch_hgt_config(run_cfg, retry=0)
    first_summary = summary_path(first_config)
    if first_summary.exists():
        return {"run": run_cfg["name"], "status": "skipped", "summary": str(first_summary)}
    last_log = None
    for retry in range(OOM_RETRIES + 1):
        cfg_path, _ = patch_hgt_config(run_cfg, retry=retry)
        code, log_path = train_process(run_cfg, cfg_path, gpu_id, retry)
        last_log = log_path
        if code == 0:
            return {"run": run_cfg["name"], "status": "ok", "summary": str(summary_path(cfg_path))}
        if not log_has_oom(log_path) or retry == OOM_RETRIES:
            break
        print("OOM detected; retry with smaller batch_seed_flows.")
    if ALLOW_CPU_FALLBACK:
        print("CPU fallback is enabled but not recommended for Kaggle HGT runs.")
    raise RuntimeError(f"HGT run failed: {run_cfg['name']}. See {last_log}")


def train_all() -> list[dict[str, Any]]:
    gpu_queue: queue.Queue[int] = queue.Queue()
    for gpu_id in GPU_IDS[:MAX_PARALLEL_HGT_RUNS]:
        gpu_queue.put(gpu_id)
    results = []
    def worker(run_cfg: dict[str, str]) -> dict[str, Any]:
        gpu_id = gpu_queue.get()
        try:
            return train_one(run_cfg, gpu_id)
        finally:
            gpu_queue.put(gpu_id)
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_HGT_RUNS) as executor:
        futures = [executor.submit(worker, run_cfg) for run_cfg in RUNS]
        for future in as_completed(futures):
            result = future.result()
            results.append(result)
            print("DONE:", result)
    return results


train_results = train_all()
train_results


In [ ]:
def build_comparison() -> list[dict[str, Any]]:
    rows = []
    for run_cfg in RUNS:
        cfg_path, _ = patch_hgt_config(run_cfg, retry=0)
        sp = summary_path(cfg_path)
        if not sp.exists():
            continue
        data = json.loads(sp.read_text(encoding="utf-8"))
        cfg = data["config"]
        model = cfg["model"]
        train = cfg["train"]
        best_val = data.get("best_val_metrics", {})
        best_test = data.get("best_test_metrics", {})
        rows.append({
            "run_name": run_cfg["name"],
            "run_dir": str(sp.parent.relative_to(WORK_DIR)),
            "hidden_dim": model["hidden_dim"],
            "num_layers": model["num_layers"],
            "num_heads": model["num_heads"],
            "batch_mode": train.get("batch_mode"),
            "batch_seed_flows": train.get("batch_seed_flows"),
            "grad_accum_steps": train.get("grad_accum_steps"),
            "best_epoch": data.get("best_epoch"),
            "val_macro_f1": best_val.get("macro_f1"),
            "test_macro_f1": best_test.get("macro_f1"),
            "test_accuracy": best_test.get("accuracy"),
            "device": data.get("device"),
        })
    out_dir = WORK_DIR / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    if rows:
        csv_path = out_dir / "hgt_existing_graph_comparison.csv"
        with csv_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
        print("Comparison CSV:", csv_path)
    return rows


def bundle_results() -> Path:
    if RESULT_ZIP.exists():
        RESULT_ZIP.unlink()
    include_roots = [WORK_DIR / "outputs", GRAPH_META_JSON, GRAPH_STORE_ROOT / "manifest.json", WORK_DIR / "configs" / "hgt_t082_k5_l3_d01.yaml", WORK_DIR / "configs" / "hgt_paper_variants"]
    with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
        for root in include_roots:
            if not root.exists():
                continue
            if root.is_file():
                try:
                    arcname = root.relative_to(WORK_DIR)
                except ValueError:
                    arcname = Path(root.name)
                zf.write(root, arcname)
                continue
            for path in root.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(WORK_DIR))
    print("Bundle:", RESULT_ZIP)
    print("Size MB:", round(RESULT_ZIP.stat().st_size / 1024 / 1024, 2))
    return RESULT_ZIP


comparison_rows = build_comparison()
bundle_results()
comparison_rows


## Notes

- Required Kaggle input (3 files):
  1. `graph_artifact_3tier_t082_k5.npz` — the three-tier graph
  2. `graph_artifact_3tier_t082_k5.meta.json` — graph metadata
  3. `graph_artifact_3tier_t082_k5_packet_semantic_x.npy.zst` — Zstandard-compressed packet-semantic feature matrix (or uncompressed `.npy`)
- This notebook intentionally skips teacher generation, student CNN training, student embedding export, and three-tier graph rebuild.
- `graph_store_v1` may be runtime JSONL. HGT training requires `layout=numpy_memmap_csr`; this notebook writes/uses `graph_store_mmap_csr_v1`.
- Default `HGT_RUN_MODE="deployment"` trains one production config. Use `HGT_RUN_MODE="paper_variants"` to train multiple configs and use both GPUs in parallel.
- `GITHUB_PULL_IF_CLONED=True` (default): when WORK_DIR already exists as a git clone, runs `git pull --ff-only` to pick up latest code changes before training.
- **Compressed sidecar support**: the notebook auto-detects and decompresses `.npy.zst` (Zstandard, preferred), `.npy.gz` (gzip), or `.npy.zip` (zip) variants. The decompressed `.npy` is written to `/tmp` (configurable via `GRAPH_PKT_SEM_DECOMPRESS_DIR`) so it does **not** consume the 19 GB `/kaggle/working` quota. The `zstandard` package is installed automatically inside `install_repo()` if not already present on the runtime.
- `GRAPH_PKT_SEM_NPY_ZST_NAME`: filename of the compressed sidecar searched in `/kaggle/input`. Change this variable in cell 1 if your upload uses a different filename.
